# Tahap 7 — Missing Value Handling

**Tujuan notebook**
Menangani missing value pada `05_integrated_dataset.csv` sesuai tiga kategori hasil audit
Tahap 6 (`quality_issue_log.csv`), tanpa mengubah jumlah baris.

**Kategori missing (dari Tahap 6):**
- **Kategori A (Ogimet)** — `rr`, `tavg`, `rh` → **interpolasi time-series**
- **Kategori B (SounderPy, `SELECTED` tetapi kosong)** — `cin`, `kindex`, `li`, `tt`,
  `sweat`, `cape` → **imputasi median bulanan** (group by bulan kalender, 1–12)
- **Kategori C (`NO_SOUNDING`)** — seluruh variabel atmosfer **dibiarkan NaN**, tidak
  diimputasi

**Deliverable:**
- `07_missing_handled.csv`
- `STAGE7_REPORT.md`


## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


## 2. Konfigurasi

In [2]:
INPUT_PATH = Path("05_integrated_dataset.csv")
OUTPUT_CSV_PATH = Path("07_missing_handled.csv")
OUTPUT_REPORT_PATH = Path("STAGE7_REPORT.md")

OGIMET_VARS = ["rr", "tavg", "rh"]
SOUNDERPY_VARS = ["cin", "kindex", "li", "tt", "sweat", "cape"]


## 3. Load Dataset & Catat Missing Asli

Masking missing asli dicatat **sebelum** proses imputasi apa pun dilakukan, karena masking
ini menjadi dasar penentuan kolom audit `missing_group` dan penghitungan jumlah nilai yang
diimputasi.

In [3]:
df = pd.read_csv(INPUT_PATH)
df["date"] = pd.to_datetime(df["date"], errors="raise")
df = df.sort_values("date").reset_index(drop=True)

n_rows_before = len(df)
n_cols_before = len(df.columns)

original_missing_ogimet = df[OGIMET_VARS].isna()
original_missing_sounderpy = df[SOUNDERPY_VARS].isna()

print(f"Shape awal: {df.shape}")
df.head()


Shape awal: (2922, 12)


,date,selected_hour,selection_status,rr,tavg,rh,cin,kindex,li,tt,sweat,cape
0,2017-01-01,12Z,SELECTED,NaN,28.2,83.9,-4.906,34.4,-4.510,42.2,219.162,2226.401
1,2017-01-02,12Z,SELECTED,26.9,26.5,87.4,-22.887,35.2,-2.955,41.5,223.381,921.164
2,2017-01-03,12Z,SELECTED,78.0,26.2,87.6,0.000,36.4,-3.317,40.5,277.368,1671.572
3,2017-01-04,12Z,SELECTED,33.0,24.7,92.4,-4.865,35.1,-0.752,39.0,294.560,397.409
4,2017-01-05,12Z,SELECTED,83.0,25.7,88.3,-15.811,38.0,-5.124,44.1,299.131,2064.721


## 4. Rule 3 — Interpolasi Time-Series untuk Ogimet (`rr`, `tavg`, `rh`)

Interpolasi dilakukan berbasis waktu (`method="time"`) menggunakan `date` sebagai indeks,
karena master calendar sudah lengkap harian tanpa celah. `limit_direction="both"` dipakai
agar missing di ujung awal/akhir deret (bukan hanya di tengah) tetap terisi — jika hanya
satu arah, titik pertama/terakhir yang missing tidak akan pernah terisi oleh interpolasi
murni.

In [4]:
def interpolate_ogimet(df: pd.DataFrame, variables: list) -> pd.DataFrame:
    """Interpolasi time-series untuk variabel Ogimet, berbasis index tanggal."""
    df = df.copy()
    indexed = df.set_index("date")
    for col in variables:
        indexed[col] = indexed[col].interpolate(method="time", limit_direction="both")
    df[variables] = indexed[variables].reset_index(drop=True)
    return df


df = interpolate_ogimet(df, OGIMET_VARS)

n_missing_ogimet_after = df[OGIMET_VARS].isna().sum().sum()
print(f"Missing Ogimet setelah interpolasi: {n_missing_ogimet_after}")


Missing Ogimet setelah interpolasi: 0


## 5. Rule 4–5 — Imputasi Median Bulanan untuk SounderPy (hanya `SELECTED`)

Median dihitung per bulan kalender (1–12, digabung lintas tahun 2017–2024) menggunakan
nilai yang **sudah tersedia** pada baris `SELECTED` di bulan tersebut. Baris
`NO_SOUNDING` sama sekali tidak disentuh (Rule 5) — nilainya tetap NaN.

In [5]:
def impute_sounderpy_monthly_median(df: pd.DataFrame, variables: list) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Imputasi median bulanan untuk variabel SounderPy, hanya pada baris SELECTED."""
    df = df.copy()
    df["_month"] = df["date"].dt.month

    is_selected = df["selection_status"] == "SELECTED"

    monthly_median_rows = []
    for col in variables:
        # Median dihitung dari baris SELECTED yang nilainya tersedia (bukan NaN).
        monthly_median = (
            df.loc[is_selected, ["_month", col]]
            .dropna(subset=[col])
            .groupby("_month")[col]
            .median()
        )
        monthly_median_rows.append(monthly_median.rename(col))

        needs_imputation = is_selected & df[col].isna()
        fill_values = df.loc[needs_imputation, "_month"].map(monthly_median)
        df.loc[needs_imputation, col] = fill_values

    monthly_median_table = pd.concat(monthly_median_rows, axis=1).sort_index()
    df = df.drop(columns=["_month"])
    return df, monthly_median_table


df, monthly_median_table = impute_sounderpy_monthly_median(df, SOUNDERPY_VARS)

n_missing_selected_after = df.loc[df["selection_status"] == "SELECTED", SOUNDERPY_VARS].isna().sum().sum()
print(f"Missing SounderPy pada SELECTED setelah imputasi: {n_missing_selected_after}")
monthly_median_table.round(3)


Missing SounderPy pada SELECTED setelah imputasi: 0


,cin,kindex,li,tt,sweat,cape
_month,,,,,,
1,-17.490,34.00,-4.370,43.40,210.612,1921.012
2,-20.367,35.20,-4.721,43.80,211.751,1832.978
3,-21.210,34.80,-4.263,43.70,212.447,1798.672
4,-22.663,35.30,-4.349,43.90,219.965,1802.594
5,-27.652,34.75,-4.020,43.60,222.694,1614.864
6,-33.665,34.00,-3.914,43.45,204.396,1168.509
7,-21.157,32.90,-4.120,43.30,205.101,1519.458
8,-23.810,33.90,-3.658,43.60,208.946,1203.952
9,-27.880,33.90,-3.314,43.20,207.991,1094.522


## 6. Rule 5 — Verifikasi `NO_SOUNDING` Tetap NaN

Memastikan tidak ada satu pun nilai atmosfer pada baris `NO_SOUNDING` yang tersentuh oleh
proses imputasi manapun di atas.

In [6]:
is_no_sounding = df["selection_status"] == "NO_SOUNDING"
n_no_sounding_still_nan = df.loc[is_no_sounding, SOUNDERPY_VARS].isna().sum().sum()
n_no_sounding_rows = int(is_no_sounding.sum())
n_expected_nan = n_no_sounding_rows * len(SOUNDERPY_VARS)

assert n_no_sounding_still_nan == n_expected_nan, (
    f"Ditemukan nilai atmosfer pada baris NO_SOUNDING yang tidak lagi NaN "
    f"({n_no_sounding_still_nan} dari {n_expected_nan} diharapkan)."
)
print(f"Terverifikasi: seluruh {n_expected_nan} sel atmosfer pada {n_no_sounding_rows} baris NO_SOUNDING tetap NaN.")


Terverifikasi: seluruh 888 sel atmosfer pada 148 baris NO_SOUNDING tetap NaN.


## 7. Rule 6 — Kolom Audit `missing_group`

Karena satu baris hanya boleh mendapat satu label, digunakan urutan prioritas berikut
(dari yang paling dominan):

1. `NO_SOUNDING` — baris berstatus `NO_SOUNDING` (ciri struktural paling menentukan)
2. `IMPUTED_SOUNDERPY` — baris `SELECTED` yang salah satu variabel atmosfernya semula
   kosong dan baru saja diimputasi
3. `IMPUTED_OGIMET` — baris yang salah satu variabel Ogimet-nya semula kosong dan baru
   saja diinterpolasi (dan tidak termasuk dua kondisi di atas)
4. `ORIGINAL` — tidak ada missing sama sekali pada baris tersebut sebelum Tahap 7

In [7]:
def assign_missing_group(df: pd.DataFrame, orig_missing_ogimet: pd.DataFrame, orig_missing_sounderpy: pd.DataFrame) -> pd.Series:
    """Tentukan missing_group per baris berdasarkan urutan prioritas yang didefinisikan."""
    had_missing_ogimet = orig_missing_ogimet.any(axis=1)
    had_missing_sounderpy_selected = orig_missing_sounderpy.any(axis=1) & (df["selection_status"] == "SELECTED")
    is_no_sounding = df["selection_status"] == "NO_SOUNDING"

    group = pd.Series("ORIGINAL", index=df.index, dtype="object")
    group = group.mask(had_missing_ogimet, "IMPUTED_OGIMET")
    group = group.mask(had_missing_sounderpy_selected, "IMPUTED_SOUNDERPY")
    group = group.mask(is_no_sounding, "NO_SOUNDING")
    return group


df["missing_group"] = assign_missing_group(df, original_missing_ogimet, original_missing_sounderpy)
df["missing_group"].value_counts()


missing_group
ORIGINAL             2707
NO_SOUNDING           148
IMPUTED_SOUNDERPY      46
IMPUTED_OGIMET         21
Name: count, dtype: int64

## 8. Validasi Integritas Struktural

Jumlah baris dan kolom asli (di luar `missing_group`) tidak boleh berubah; tidak boleh
ada duplicate date.

In [8]:
assert len(df) == n_rows_before, f"Jumlah baris berubah: {len(df)} vs {n_rows_before}"
assert len(df.columns) == n_cols_before + 1, "Jumlah kolom tidak sesuai ekspektasi (+1 untuk missing_group)."

n_duplicate_date = int(df["date"].duplicated().sum())
assert n_duplicate_date == 0, f"Ditemukan duplicate date: {n_duplicate_date}"

print(f"Total row : {len(df)} (tetap {n_rows_before})")
print(f"Total kolom: {len(df.columns)}")
print(f"Duplicate date: {n_duplicate_date}")


Total row : 2922 (tetap 2922)
Total kolom: 13
Duplicate date: 0


## 9. Susun Kolom Output & Simpan CSV

In [9]:
OUTPUT_COLUMNS = [
    "date", "selected_hour", "selection_status",
    "rr", "tavg", "rh",
    "cin", "kindex", "li", "tt", "sweat", "cape",
    "missing_group",
]
df = df[OUTPUT_COLUMNS]

df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Tersimpan: {OUTPUT_CSV_PATH.resolve()}")
df.head()


Tersimpan: /home/claude/work/07_missing_handled.csv


,date,selected_hour,selection_status,rr,tavg,rh,cin,kindex,li,tt,sweat,cape,missing_group
0,2017-01-01,12Z,SELECTED,26.9,28.2,83.9,-4.906,34.4,-4.510,42.2,219.162,2226.401,IMPUTED_OGIMET
1,2017-01-02,12Z,SELECTED,26.9,26.5,87.4,-22.887,35.2,-2.955,41.5,223.381,921.164,ORIGINAL
2,2017-01-03,12Z,SELECTED,78.0,26.2,87.6,0.000,36.4,-3.317,40.5,277.368,1671.572,ORIGINAL
3,2017-01-04,12Z,SELECTED,33.0,24.7,92.4,-4.865,35.1,-0.752,39.0,294.560,397.409,ORIGINAL
4,2017-01-05,12Z,SELECTED,83.0,25.7,88.3,-15.811,38.0,-5.124,44.1,299.131,2064.721,ORIGINAL


## 10. Audit Wajib

Jumlah nilai yang diimputasi, jumlah nilai yang tetap NaN, dan distribusi
`missing_group`.

In [10]:
def compute_stage7_audit(
    df: pd.DataFrame,
    orig_missing_ogimet: pd.DataFrame,
    orig_missing_sounderpy: pd.DataFrame,
) -> dict:
    """Hitung ringkasan audit wajib Tahap 7."""
    is_selected = df["selection_status"] == "SELECTED"

    n_imputed_ogimet = int(orig_missing_ogimet.to_numpy().sum())
    n_imputed_sounderpy = int((orig_missing_sounderpy.to_numpy() & is_selected.to_numpy()[:, None]).sum())
    n_total_imputed = n_imputed_ogimet + n_imputed_sounderpy

    n_still_nan = int(df[SOUNDERPY_VARS].isna().to_numpy().sum())

    imputed_per_ogimet_var = orig_missing_ogimet.sum().to_dict()
    imputed_per_sounderpy_var = {
        col: int((orig_missing_sounderpy[col] & is_selected).sum()) for col in SOUNDERPY_VARS
    }

    return {
        "n_imputed_ogimet": n_imputed_ogimet,
        "n_imputed_sounderpy": n_imputed_sounderpy,
        "n_total_imputed": n_total_imputed,
        "n_still_nan": n_still_nan,
        "imputed_per_ogimet_var": imputed_per_ogimet_var,
        "imputed_per_sounderpy_var": imputed_per_sounderpy_var,
        "missing_group_distribution": df["missing_group"].value_counts().to_dict(),
    }


audit_result = compute_stage7_audit(df, original_missing_ogimet, original_missing_sounderpy)

print(f"Jumlah nilai diimputasi (Ogimet)   : {audit_result['n_imputed_ogimet']}")
print(f"Jumlah nilai diimputasi (SounderPy): {audit_result['n_imputed_sounderpy']}")
print(f"Total nilai diimputasi              : {audit_result['n_total_imputed']}")
print(f"Jumlah nilai yang tetap NaN          : {audit_result['n_still_nan']}")
print("Distribusi missing_group:")
for grp, jumlah in audit_result["missing_group_distribution"].items():
    print(f"  {grp}: {jumlah}")


Jumlah nilai diimputasi (Ogimet)   : 85
Jumlah nilai diimputasi (SounderPy): 186
Total nilai diimputasi              : 271
Jumlah nilai yang tetap NaN          : 888
Distribusi missing_group:
  ORIGINAL: 2707
  NO_SOUNDING: 148
  IMPUTED_SOUNDERPY: 46
  IMPUTED_OGIMET: 21


## 11. Cek Acceptance Criteria

In [11]:
def check_acceptance_criteria(df: pd.DataFrame, audit: dict, n_rows_before: int) -> pd.DataFrame:
    """Bandingkan hasil aktual terhadap acceptance criteria Tahap 7."""
    n_duplicate_date = int(df["date"].duplicated().sum())
    n_missing_ogimet = int(df[OGIMET_VARS].isna().sum().sum())
    n_missing_sounderpy_selected = int(
        df.loc[df["selection_status"] == "SELECTED", SOUNDERPY_VARS].isna().sum().sum()
    )
    n_no_sounding_preserved = int(
        df.loc[df["selection_status"] == "NO_SOUNDING", SOUNDERPY_VARS].isna().sum().sum()
    )
    n_no_sounding_expected = int((df["selection_status"] == "NO_SOUNDING").sum()) * len(SOUNDERPY_VARS)

    rows = [
        ("total row tetap 2922", len(df) == 2922 == n_rows_before, len(df)),
        ("duplicate date tetap 0", n_duplicate_date == 0, n_duplicate_date),
        ("missing Ogimet = 0", n_missing_ogimet == 0, n_missing_ogimet),
        ("missing SounderPy pada SELECTED = 0", n_missing_sounderpy_selected == 0, n_missing_sounderpy_selected),
        (
            "missing NO_SOUNDING tetap dipertahankan",
            n_no_sounding_preserved == n_no_sounding_expected,
            n_no_sounding_preserved,
        ),
    ]
    return pd.DataFrame(rows, columns=["kriteria", "terpenuhi", "nilai_aktual"])


acceptance_df = check_acceptance_criteria(df, audit_result, n_rows_before)
acceptance_df


,kriteria,terpenuhi,nilai_aktual
0,total row tetap 2922,True,2922
1,duplicate date tetap 0,True,0
2,missing Ogimet = 0,True,0
3,missing SounderPy pada SELECTED = 0,True,0
4,missing NO_SOUNDING tetap dipertahankan,True,888


## 12. Susun `STAGE7_REPORT.md`

In [12]:
def fmt_table(df: pd.DataFrame) -> str:
    """Format DataFrame kecil menjadi tabel Markdown sederhana."""
    return df.to_markdown(index=False)


def build_stage7_report(
    audit: dict,
    acceptance_df: pd.DataFrame,
    monthly_median_table: pd.DataFrame,
) -> str:
    """Bangun konten STAGE7_REPORT.md dari hasil audit dan validasi aktual."""
    ogimet_lines = "\n".join(
        f"- {var}: {jumlah}" for var, jumlah in audit["imputed_per_ogimet_var"].items()
    )
    sounderpy_lines = "\n".join(
        f"- {var}: {jumlah}" for var, jumlah in audit["imputed_per_sounderpy_var"].items()
    )
    group_lines = "\n".join(
        f"- {grp}: {jumlah}" for grp, jumlah in audit["missing_group_distribution"].items()
    )
    acceptance_lines = "\n".join(
        f"- [{'x' if row.terpenuhi else ' '}] {row.kriteria} (aktual: {row.nilai_aktual})"
        for row in acceptance_df.itertuples(index=False)
    )

    report = f"""# STAGE7_REPORT — Missing Value Handling

## Ringkasan

- Jumlah nilai diimputasi (Ogimet, interpolasi time-series): {audit['n_imputed_ogimet']}
- Jumlah nilai diimputasi (SounderPy, median bulanan, SELECTED): {audit['n_imputed_sounderpy']}
- Total nilai diimputasi: {audit['n_total_imputed']}
- Jumlah nilai yang tetap NaN (NO_SOUNDING, sengaja tidak diimputasi): {audit['n_still_nan']}

## Rincian Imputasi per Variabel Ogimet (interpolasi time-series)

{ogimet_lines}

## Rincian Imputasi per Variabel SounderPy (median bulanan, hanya SELECTED)

{sounderpy_lines}

## Tabel Median Bulanan yang Digunakan (SounderPy)

{fmt_table(monthly_median_table.round(3).reset_index().rename(columns={"_month": "bulan"}))}

## Distribusi missing_group

{group_lines}

**Urutan prioritas label `missing_group`** (satu baris hanya mendapat satu label):
1. `NO_SOUNDING` — baris berstatus NO_SOUNDING
2. `IMPUTED_SOUNDERPY` — baris SELECTED dengan variabel atmosfer yang semula kosong
3. `IMPUTED_OGIMET` — baris dengan variabel Ogimet yang semula kosong (dan bukan kondisi 1–2)
4. `ORIGINAL` — tidak ada missing sama sekali sebelum Tahap 7

## Metodologi

- **Ogimet (`rr`, `tavg`, `rh`)**: interpolasi time-series (`pandas.interpolate(method="time")`)
  berbasis index tanggal harian lengkap. `limit_direction="both"` digunakan agar missing di
  ujung awal/akhir deret turut terisi.
- **SounderPy (`cin`, `kindex`, `li`, `tt`, `sweat`, `cape`) pada baris `SELECTED`**:
  imputasi median bulanan — median dihitung per bulan kalender (1–12, digabung lintas
  tahun) dari nilai yang tersedia pada baris `SELECTED` di bulan yang sama.
- **SounderPy pada baris `NO_SOUNDING`**: tidak diimputasi sama sekali, tetap NaN.

## Acceptance Criteria

{acceptance_lines}
- [x] seluruh proses terdokumentasi (notebook ini + laporan ini)

## Catatan Cakupan

Tidak ada baris yang dihapus dan jumlah baris tidak berubah. Tidak dilakukan outlier
removal, labeling, feature engineering, scaling, atau train/test split pada tahap ini.
Dataset ini adalah input untuk tahap berikutnya dalam workflow.
"""
    return report


report_content = build_stage7_report(audit_result, acceptance_df, monthly_median_table)
OUTPUT_REPORT_PATH.write_text(report_content)
print(f"Tersimpan: {OUTPUT_REPORT_PATH.resolve()}")


Tersimpan: /home/claude/work/STAGE7_REPORT.md
